In [1]:
import pandas as pd
import numpy as np
import math

df = pd.read_csv("play_tennis.csv")

# ---------- ENTROPY FUNCTION ----------
def entropy(col):
    elements, counts = np.unique(col, return_counts=True)
    ent = 0
    for i in range(len(elements)):
        p = counts[i] / sum(counts)
        ent += -p * math.log2(p)
    return ent

# ---------- INFORMATION GAIN ----------
def info_gain(data, feature, target):
    total_entropy = entropy(data[target])

    vals, counts = np.unique(data[feature], return_counts=True)
    weighted_entropy = 0

    for i in range(len(vals)):
        subset = data[data[feature] == vals[i]]
        weighted_entropy += (counts[i] / sum(counts)) * entropy(subset[target])

    gain = total_entropy - weighted_entropy

    print(f"\nFeature: {feature}")
    print(f"  Total Entropy: {total_entropy:.4f}")
    print(f"  Weighted Entropy: {weighted_entropy:.4f}")
    print(f"  Information Gain: {gain:.4f}")

    return gain

# ---------- ID3 WITH PRINTING ----------
def id3(data, features, target, level=0):
    indent = "    " * level

    # If only one class → leaf node
    if len(np.unique(data[target])) == 1:
        leaf = np.unique(data[target])[0]
        print(indent + f"Leaf: {leaf}")
        return leaf

    # If no features left → majority class
    if len(features) == 0:
        majority = data[target].mode()[0]
        print(indent + f"Leaf: {majority}")
        return majority

    print("\n" + indent + "-------------------------------------")
    print(indent + f"Dataset:\n{data}")
    print(indent + "-------------------------------------")

    # Compute information gain for each feature
    gains = {f: info_gain(data, f, target) for f in features}

    # Pick best feature
    best_feature = max(gains, key=gains.get)
    print(indent + f"\nBest Feature Selected: {best_feature}")

    tree = {best_feature: {}}
    remaining_features = [f for f in features if f != best_feature]

    # Split on each value of the best feature
    for value in np.unique(data[best_feature]):
        print(indent + f"\nSplitting on {best_feature} = {value}")
        subset = data[data[best_feature] == value]

        tree[best_feature][value] = id3(subset, remaining_features, target, level+1)

    return tree


# ---------- RUN THE ALGORITHM ----------
features = list(df.columns)
features.remove("day")   # remove index column
features.remove("play")  # target

decision_tree = id3(df, features, "play")

print("\n\nFINAL DECISION TREE:")
print(decision_tree)





-------------------------------------
Dataset:
    day   outlook  temp humidity    wind play
0    D1     Sunny   Hot     High    Weak   No
1    D2     Sunny   Hot     High  Strong   No
2    D3  Overcast   Hot     High    Weak  Yes
3    D4      Rain  Mild     High    Weak  Yes
4    D5      Rain  Cool   Normal    Weak  Yes
5    D6      Rain  Cool   Normal  Strong   No
6    D7  Overcast  Cool   Normal  Strong  Yes
7    D8     Sunny  Mild     High    Weak   No
8    D9     Sunny  Cool   Normal    Weak  Yes
9   D10      Rain  Mild   Normal    Weak  Yes
10  D11     Sunny  Mild   Normal  Strong  Yes
11  D12  Overcast  Mild     High  Strong  Yes
12  D13  Overcast   Hot   Normal    Weak  Yes
13  D14      Rain  Mild     High  Strong   No
-------------------------------------

Feature: outlook
  Total Entropy: 0.9403
  Weighted Entropy: 0.6935
  Information Gain: 0.2467

Feature: temp
  Total Entropy: 0.9403
  Weighted Entropy: 0.9111
  Information Gain: 0.0292

Feature: humidity
  Total Entropy: